# Assignment 2.3 — DDIM Sampling

Reuse the **non-conditional** model trained in Assignment 2.2 and replace DDPM sampling with **DDIM**.

**Kaggle Tesla P100:** run **Cell 1 once**, then use **Restart Session**, then **Run All**.

No retraining. No bonus. PyTorch only.


In [ ]:
# Cell 1 — P100 environment setup
# Run ONCE, then Restart Session before running the remaining cells.
# Do not import torch in this cell.

import sys
import subprocess
import importlib.metadata as metadata

current = metadata.version("torch")
print("Current installed torch:", current)

if not current.startswith("2.7.1"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q",
        "torch==2.7.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu126"
    ])

    print("\nPyTorch 2.7.1 + cu126 installed.")
    print("NOW: Restart Session, then Run All.")
else:
    print("P100-compatible PyTorch is already installed.")


In [ ]:
# Cell 2 — Imports and verify P100 support
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("Supported arch:", torch.cuda.get_arch_list())

    assert "sm_60" in torch.cuda.get_arch_list(), (
        "P100 needs sm_60. Run Cell 1, Restart Session, then Run All."
    )


In [ ]:
# Cell 3 — Same settings as Assignment 2.2
T = 200
schedule_type = "cosine"

base_channels = 32
time_dim = 128

sampling_steps = 50


In [ ]:
# Cell 4 — Same noise scheduler
class NoiseScheduler:
    def __init__(self, T=200, schedule="linear", device="cpu"):
        if schedule == "linear":
            betas = torch.linspace(1e-4, 0.02, T)

        elif schedule == "cosine":
            s = 0.008
            steps = torch.arange(T + 1, dtype=torch.float32)

            alpha_bar = torch.cos(
                ((steps / T + s) / (1 + s)) * math.pi / 2
            ) ** 2

            alpha_bar = alpha_bar / alpha_bar[0]

            betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
            betas = torch.clamp(betas, 1e-4, 0.999)

        else:
            raise ValueError("schedule must be 'linear' or 'cosine'")

        self.beta = betas.to(device)
        self.alpha = 1.0 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)


scheduler = NoiseScheduler(
    T=T,
    schedule=schedule_type,
    device=device
)


In [ ]:
# Cell 5 — Same U-Net as Assignment 2.2
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        scale = math.log(10000) / (half - 1)

        freq = torch.exp(
            torch.arange(half, device=t.device) * -scale
        )

        angles = t.float().unsqueeze(1) * freq.unsqueeze(0)

        return torch.cat([
            torch.sin(angles),
            torch.cos(angles)
        ], dim=1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)

        self.time_proj = nn.Linear(time_dim, out_ch)

        self.residual = (
            nn.Conv2d(in_ch, out_ch, 1)
            if in_ch != out_ch
            else nn.Identity()
        )

    def forward(self, x, t_emb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = F.silu(self.norm2(self.conv2(h)))

        return h + self.residual(x)


class SimpleUNet(nn.Module):
    def __init__(self, base_channels=32, time_dim=128):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.input_conv = nn.Conv2d(1, base_channels, 3, padding=1)

        self.down1 = ResBlock(
            base_channels, base_channels, time_dim
        )

        self.downsample1 = nn.Conv2d(
            base_channels, base_channels * 2,
            4, stride=2, padding=1
        )

        self.down2 = ResBlock(
            base_channels * 2,
            base_channels * 2,
            time_dim
        )

        self.downsample2 = nn.Conv2d(
            base_channels * 2, base_channels * 4,
            4, stride=2, padding=1
        )

        self.middle = ResBlock(
            base_channels * 4, base_channels * 4, time_dim
        )

        self.upsample1 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2,
            4, stride=2, padding=1
        )

        self.up1 = ResBlock(
            base_channels * 4, base_channels * 2, time_dim
        )

        self.upsample2 = nn.ConvTranspose2d(
            base_channels * 2, base_channels,
            4, stride=2, padding=1
        )

        self.up2 = ResBlock(
            base_channels * 2, base_channels, time_dim
        )

        self.output_conv = nn.Conv2d(base_channels, 1, 1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)

        x = self.input_conv(x)

        skip1 = self.down1(x, t_emb)

        x = self.downsample1(skip1)
        skip2 = self.down2(x, t_emb)

        x = self.downsample2(skip2)
        x = self.middle(x, t_emb)

        x = self.upsample1(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.up1(x, t_emb)

        x = self.upsample2(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.up2(x, t_emb)

        return self.output_conv(x)


In [ ]:
# Cell 6 — Find and load Assignment 2.2 checkpoint
matches = list(Path("/kaggle/input").rglob("diffusion_mnist.pth"))

if not matches:
    raise FileNotFoundError(
        "diffusion_mnist.pth not found. "
        "Upload the NON-CONDITIONAL checkpoint from Assignment 2.2."
    )

checkpoint_path = matches[0]
print("Checkpoint:", checkpoint_path)

model = SimpleUNet(
    base_channels=base_channels,
    time_dim=time_dim
).to(device)

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model.eval()

print("Model loaded.")


In [ ]:
# Cell 7 — One DDIM step
@torch.no_grad()
def ddim_step(model, x_t, t, prev_t):
    t_batch = torch.full(
        (x_t.size(0),),
        t,
        device=device,
        dtype=torch.long
    )

    eps_pred = model(x_t, t_batch)

    alpha_bar_t = scheduler.alpha_bar[t]

    x0_pred = (
        x_t
        - torch.sqrt(1.0 - alpha_bar_t) * eps_pred
    ) / torch.sqrt(alpha_bar_t)

    x0_pred = x0_pred.clamp(-1, 1)

    if prev_t < 0:
        return x0_pred

    alpha_bar_prev = scheduler.alpha_bar[prev_t]

    return (
        torch.sqrt(alpha_bar_prev) * x0_pred
        + torch.sqrt(1.0 - alpha_bar_prev) * eps_pred
    )


In [ ]:
# Cell 8 — DDIM sampling loop
@torch.no_grad()
def sample_ddim(model, n_samples=16, sampling_steps=50):
    x = torch.randn(
        n_samples, 1, 28, 28,
        device=device
    )

    timesteps = torch.linspace(
        T - 1,
        0,
        sampling_steps
    ).round().long().tolist()

    timesteps = list(dict.fromkeys(timesteps))

    for i, t in enumerate(timesteps):
        prev_t = (
            -1
            if i == len(timesteps) - 1
            else timesteps[i + 1]
        )

        x = ddim_step(model, x, t, prev_t)

    return ((x.clamp(-1, 1) + 1) / 2).cpu()


In [ ]:
# Cell 9 — Generate samples
generated = sample_ddim(
    model,
    n_samples=16,
    sampling_steps=sampling_steps
)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))

for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle(f"DDIM Samples ({sampling_steps} steps)")
plt.tight_layout()
plt.show()
